In [2]:
import pandas as pd
from functools import partial

In [3]:
def build_panel_dataset(
    df,
    freq="W",
    min_obs=5,
    indiv_col="user",
    date_col="date",
    metrics=None
):
    """
    Construit un dataset panel agrégé individu x période.

    Paramètres
    ----------
    df : DataFrame source

    freq : str
        Fréquence temporelle pandas ("W", "M", etc.)

    min_obs : int
        Nombre minimal d'observations par individu/période

    indiv_col : str
        Colonne identifiant individu

    date_col : str
        Colonne datetime

    metrics : dict
        Dictionnaire :
        {
            "nom_variable_finale": fonction
        }

        où chaque fonction prend en entrée
        le sous-dataframe du groupe individu x période.

    Retour
    ------
    panel_df : DataFrame agrégé
    """

    # =========================================================
    # Préparation
    # =========================================================

    data = df.copy()

    data[date_col] = pd.to_datetime(data[date_col])

    data["period"] = data[date_col].dt.to_period(freq)

    # =========================================================
    # Taille des groupes
    # =========================================================

    counts = (
        data
        .groupby([indiv_col, "period"])
        .size()
        .reset_index(name="n_obs")
    )

    # Groupes valides
    valid_groups = counts[counts["n_obs"] >= min_obs]

    # Garde seulement les groupes valides
    data = data.merge(
        valid_groups[[indiv_col, "period", "n_obs"]],
        on=[indiv_col, "period"],
        how="inner"
    )

    # =========================================================
    # Métriques personnalisées
    # =========================================================

    if metrics is None:
        metrics = {}

    rows = []

    grouped = data.groupby([indiv_col, "period"])

    for (user, period), group in grouped:

        row = {
            indiv_col: user,
            "period": period,
            "n_obs": len(group)
        }

        # Calcul des métriques custom
        for metric_name, metric_func in metrics.items():

            try:
                row[metric_name] = metric_func(group)

            except Exception as e:
                row[metric_name] = pd.NA
                print(
                    f"Erreur pour {metric_name} "
                    f"({user}, {period}) : {e}"
                )

        rows.append(row)

    panel_df = pd.DataFrame(rows)

    return panel_df

In [4]:
def left_right_side(df):
    if df['user_political'].sum()<3:
        return "Apolitical"
    else :
        if len(df["user_left_right"].mode())>1:
            return "Apolitical"
        else:
            return df["user_left_right"].mode().iloc[0]

def polarisation_against_side(df, user_side):
    other_side = [l for l in ['Left', 'Right'] if l!=user_side][0]
    x = df[df['inter_left_right']==other_side]
    if len(x)==0: # If no interactions with other side, polarization is null
        return 0
    prop_violent_vs_other = x['user_violent'].sum()/len(df) 
    y = df[df['inter_left_right']==user_side]
    if len(y)==0: # If no interactions with same side, polarization is proportion of violent interactions against other side
        return prop_violent_vs_other
    prop_violent_vs_same = y['user_violent'].sum()/len(df)
    return prop_violent_vs_other - prop_violent_vs_same

def homophilie(df, user_side):
    x = df[df['inter_left_right']==user_side]
    return len(x)/len(df)

def volume_cross_interactions(df, other_side):
    x = df[df['inter_left_right']==other_side]
    return len(x)

In [5]:
metrics = {

    # Nombre d'observations politique
    "user_total_political_interactions": lambda g: g["user_political"].sum(),

    # bord politique
    "user_period_side": left_right_side,

    # Polarization
    'user_polarization_left' : partial(polarisation_against_side, user_side='Left'),
    'user_polarization_right' : partial(polarisation_against_side, user_side='Right'),

    # Homophilie
    'user_homophilie_left' : partial(homophilie, user_side='Left'),
    'user_homophilie_right' : partial(homophilie, user_side='Right'),

    # Volume cross interactions
    'volume_left_cross_right' : partial(volume_cross_interactions, other_side='Right'),
    'volume_right_cross_left' : partial(volume_cross_interactions, other_side='Left')
}

In [6]:
df = pd.read_csv('../clean_data/struct_panel.csv')
df['user_violent'] = (df['user_toxicity_level'] >= df['user_toxicity_level'].quantile(0.75)).astype(int)

In [7]:
panel_df = build_panel_dataset(
    df,
    freq="M",
    min_obs=10,
    metrics=metrics
)

C:\Users\cfrou\AppData\Local\Temp\ipykernel_32992\2148845493.py:50: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  data["period"] = data[date_col].dt.to_period(freq)


In [8]:
panel_df['user_polarization']=0
panel_df.loc[panel_df['user_period_side']=='Left', 'user_polarization'] = panel_df['user_polarization_left']
panel_df.loc[panel_df['user_period_side']=='Right', 'user_polarization'] = panel_df['user_polarization_right']


panel_df['user_homophilie']=0
panel_df.loc[panel_df['user_period_side']=='Left', 'user_homophilie'] = panel_df['user_homophilie_left']
panel_df.loc[panel_df['user_period_side']=='Right', 'user_homophilie'] = panel_df['user_homophilie_right']


panel_df['user_volume_cross_inter']=0
panel_df.loc[panel_df['user_period_side']=='Left', 'user_volume_cross_inter'] = panel_df['volume_left_cross_right']
panel_df.loc[panel_df['user_period_side']=='Right', 'user_volume_cross_inter'] = panel_df['volume_right_cross_left']

panel_df = panel_df.drop(columns=['user_polarization_left', 'user_polarization_right',
       'user_homophilie_left', 'user_homophilie_right',
       'volume_left_cross_right', 'volume_right_cross_left'])

C:\Users\cfrou\AppData\Local\Temp\ipykernel_32992\1404559989.py:2: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[ 0.         -0.01176471  0.         ... -0.04395604 -0.01675978
  0.        ]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  panel_df.loc[panel_df['user_period_side']=='Left', 'user_polarization'] = panel_df['user_polarization_left']
C:\Users\cfrou\AppData\Local\Temp\ipykernel_32992\1404559989.py:7: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[0.03092784 0.05882353 0.09090909 ... 0.16483516 0.06145251 0.18181818]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  panel_df.loc[panel_df['user_period_side']=='Left', 'user_homophilie'] = panel_df['user_homophilie_left']


In [9]:
panel_df.columns

Index(['user', 'period', 'n_obs', 'user_total_political_interactions',
       'user_period_side', 'user_polarization', 'user_homophilie',
       'user_volume_cross_inter'],
      dtype='object')

In [ ]:
# 2. Tri
panel_df = panel_df.sort_values(['user', 'period'])

# 3. Création de toutes les périodes manquantes par individu
all_periods = (
    panel_df.groupby('user')
      .apply(lambda g: pd.period_range(start=g['period'].min(),
                                       end=g['period'].max(),
                                       freq='M'))
      .explode()
      .reset_index()
      .rename(columns={0: 'period'})
)

df_full = all_periods.merge(panel_df, on=['user', 'period'], how='left')

cols_to_fill = panel_df.columns.difference(['user', 'period'])
df_full[cols_to_fill] = df_full[cols_to_fill].fillna(0)

C:\Users\cfrou\AppData\Local\Temp\ipykernel_32992\231174369.py:7: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.period_range(start=g['period'].min(),


In [11]:
len(df_full)

24476

In [24]:
# Nombre minimum d'observations par individu
k = 6

# Filtrer les individus ayant au moins k observations non manquantes
df_filtered = (
    df_full.groupby("user")
      .filter(lambda x: x["period"].notna().sum() >= k)
)

df_filtered.loc[df_filtered['user_period_side']==0, 'user_period_side'] = 'Apolitical'
len(df_filtered)


23921

In [20]:
a = 0
T = []
import numpy as np
for user in df_filtered['user'].unique():
    x = df_filtered[df_filtered['user']==user]
    T.append(len(x))
    if (x['user_polarization']==0).sum() ==len(x):
        a+=1
print(a)
print(np.mean(T))

361
34.31994261119082


In [29]:
df_filtered = df_filtered.drop(columns=['user_volume_cross_inter'])

In [35]:
df_filtered['user_Left'] = (df_filtered['user_period_side']=='Left').astype(int)
df_filtered['user_Right'] = (df_filtered['user_period_side']=='Right').astype(int)
df_filtered = df_filtered.drop(columns=['user_period_side'])

In [37]:
df_filtered.to_csv('../clean_data/panel_interactions_pVAR.csv', index=False)

In [15]:
df_full.to_csv('../clean_data/full_panel_interactions.csv', index=False)